# H1 - Query Expansion Research Pipeline

Checks whether an LLM can generate English, search-ready, faithful, diverse, and self-contained expanded queries from Vietnamese user queries.

Main workflow:

1. Initialize imports, environment variables, and experiment paths.
2. Build synthetic evaluation cases with required concepts, forbidden drift terms, and reference expansions.
3. Define intrinsic query-expansion metrics.
4. Configure models, prompt variants, output parsing, and retry behavior.
5. Run local model/prompt/hyperparameter grids and save comparison tables.
6. Optionally publish the same evaluation to LangSmith for trace inspection and experiment comparison.



## 1. Environment And Paths

This cell imports libraries, loads `.env` from the repository root, and creates stable paths for data and experiment outputs. The `find_repo_root` helper makes the notebook usable whether it is launched from the repository root or from another working directory.


In [2]:
from __future__ import annotations

import json
import math
import os
import re
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from itertools import combinations, product
from pathlib import Path
from typing import Any, Iterable, Literal

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator

from langchain_core.messages import BaseMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

try:
    from langchain_openai import ChatOpenAI
except ImportError:
    ChatOpenAI = None

try:
    from langsmith import Client, traceable
except ImportError:
    Client = None
    traceable = None

In [3]:
import yaml

In [24]:
def find_repo_root(start: Path) -> Path:
    """Find the repository root so the notebook works from Jupyter or CLI execution."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks" / "agent").exists() and (candidate / "scripts").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
load_dotenv(REPO_ROOT / ".env", override=True)
NOTEBOOK_DIR = REPO_ROOT / "notebooks" / "agent"
QUERY_DIR = REPO_ROOT / "scripts" / "query-p1-groupA"
EXPERIMENT_DIR = NOTEBOOK_DIR / "experiments" / "h1"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)


print("REPO_ROOT:", REPO_ROOT)
print("QUERY_DIR:", QUERY_DIR)
print("EXPERIMENT_DIR:", EXPERIMENT_DIR)

print(type(os.getenv("GROQ_API_KEY")) == str)
print(type(os.getenv("CEREBRAS_API_KEY")) == str)
print(type(os.getenv("NVIDIA_API_KEY")) == str)

REPO_ROOT: D:\University\Projects\Individual projects\Multimodal-Retrieval
QUERY_DIR: D:\University\Projects\Individual projects\Multimodal-Retrieval\scripts\query-p1-groupA
EXPERIMENT_DIR: D:\University\Projects\Individual projects\Multimodal-Retrieval\notebooks\agent\experiments\h1
True
True
True


## 2. Data

The generated JSONL file is saved to `notebooks/agent/experiments/h1/data/synthetic_cases.jsonl` so the same cases can be reused by local runs and LangSmith evaluation.


In [5]:
# Default number of expanded queries expected from each model call.
DEFAULT_K = 5
SYNTHETIC_DATA_PATH = os.path.join(EXPERIMENT_DIR, 'data', "synthetic_cases.jsonl")

with open(SYNTHETIC_DATA_PATH, "r", encoding="utf-8") as f:
    SYNTHETIC_CASES = [json.loads(line) for line in f]
SYNTHETIC_CASES

[{'case_id': 'kis_yellow_raincoat_motorbike_night',
  'query_type': 'KIS',
  'query': 'Tìm cảnh môt người đàn ông mặc áo mưa màu vàng chạy xe máy qua con đường ngập nước vào ban đêm.',
  'required_concepts': {'person': ['man', 'male person', 'rider'],
   'clothing': ['yellow raincoat', 'yellow poncho', 'yellow waterproof coat'],
   'vehicle': ['motorbike', 'motorcycle', 'scooter'],
   'scene': ['flooded street', 'waterlogged road', 'street flooding'],
   'time': ['night', 'nighttime', 'dark']},
  'forbidden_terms': ['taxi', 'car', 'daytime', 'snow', 'woman'],
  'reference_expansions': ['man in a yellow raincoat riding a motorcycle through a flooded street at night',
   'nighttime scene of a scooter crossing a waterlogged road with a rider wearing yellow rain gear',
   'male motorbike rider in a yellow poncho on a dark flooded urban road',
   'video frame showing motorcycle rider wearing yellow waterproof coat during street flooding at night',
   'yellow raincoat motorcycle flooded road

## 3. Data Preprocessing 

The first helpers normalize text and compute lexical overlap. The middle helpers measure concept coverage, hallucination avoidance, query diversity, English readiness, and length quality. The final function combines these into one metric dictionary for each model output.

`overall_score` is only a ranking helper for H1 experiments. It should not be interpreted as retrieval quality.


In [6]:
STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "to", "in", "on", "at", "by", "for", "with", "from",
    "into", "during", "showing", "scene", "video", "frame", "query", "search", "what", "is", "are",
}
VIETNAMESE_DIACRITICS = set("aăâeêioôơuưyAĂÂEÊIOÔƠUƯYáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵđĐ")


def normalize_text(text: str) -> str:
    """Lowercase and remove punctuation for lightweight lexical matching."""
    text = text.casefold()
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str, keep_stopwords: bool = False) -> list[str]:
    """Tokenize normalized English text, optionally preserving stopwords for n-grams."""
    tokens = re.findall(r"[a-z0-9]+", normalize_text(text))
    if keep_stopwords:
        return tokens
    return [token for token in tokens if token not in STOPWORDS]


def ngrams(tokens: list[str], n: int) -> list[tuple[str, ...]]:
    """Return contiguous n-grams used by distinct-n diversity metrics."""
    if len(tokens) < n:
        return []
    return [tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)]


def deduplicate_queries(queries: Iterable[str]) -> list[str]:
    """Normalize whitespace and remove duplicate queries while preserving order."""
    result: list[str] = []
    seen: set[str] = set()
    for item in queries:
        query = " ".join(str(item).strip().split())
        key = query.casefold()
        if query and key not in seen:
            result.append(query)
            seen.add(key)
    return result

## 4. Metric

| Metric | Ý nghĩa |
| --- | --- |
| `exact_k` | 1 nếu model trả đúng `k` query duy nhất, ngược lại 0 |
| `unique_ratio` | Tỉ lệ query không trùng lặp |
| `required_concept_coverage` | Tỉ lệ concept bắt buộc xuất hiện trong expanded queries |
| `forbidden_avoidance` | Mức độ tránh các term bị cấm |
| `reference_token_overlap` | Jaccard overlap giữa token sinh ra và reference expansions |
| `pairwise_lexical_diversity` | Độ khác nhau trung bình giữa các expanded query |
| `distinct_1`, `distinct_2` | Độ đa dạng unigram/bigram |
| `englishish_score` | Kiểm tra output có thiên về tiếng Anh/ASCII hay không |
| `length_score` | Kiểm tra query có độ dài hợp lý cho search hay không |
| `overall_score` | Điểm tổng hợp có trọng số để sort bảng so sánh |
| `query_drift_risk` | Rủi ro drift, tính từ coverage thấp hoặc forbidden hit cao |

In [7]:
def jaccard(a: set[str], b: set[str]) -> float:
    """Compute Jaccard similarity with sensible behavior for empty sets."""
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def concept_coverage(expanded_queries: list[str], required_concepts: dict[str, list[str]]) -> tuple[float, dict[str, bool]]:
    """Measure how many required semantic concepts appear in the generated queries."""
    joined = normalize_text(" ".join(expanded_queries))
    coverage: dict[str, bool] = {}
    for concept, aliases in required_concepts.items():
        coverage[concept] = any(normalize_text(alias) in joined for alias in aliases)
    if not coverage:
        return 1.0, coverage
    return sum(coverage.values()) / len(coverage), coverage


def forbidden_avoidance(expanded_queries: list[str], forbidden_terms: list[str]) -> tuple[float, list[str]]:
    """Penalize generated queries that introduce known drift/hallucination terms."""
    joined = normalize_text(" ".join(expanded_queries))
    hits = [term for term in forbidden_terms if normalize_text(term) in joined]
    if not forbidden_terms:
        return 1.0, []
    return 1.0 - (len(hits) / len(forbidden_terms)), hits


def reference_token_overlap(expanded_queries: list[str], reference_expansions: list[str]) -> float:
    """Compare generated vocabulary against human-written reference expansions."""
    generated = set(tokenize(" ".join(expanded_queries)))
    reference = set(tokenize(" ".join(reference_expansions)))
    return jaccard(generated, reference)


def pairwise_jaccard_diversity(expanded_queries: list[str]) -> float:
    """Estimate lexical diversity as one minus average pairwise Jaccard similarity."""
    pairs = list(combinations(expanded_queries, 2))
    if not pairs:
        return 0.0
    similarities = []
    for left, right in pairs:
        similarities.append(jaccard(set(tokenize(left)), set(tokenize(right))))
    return 1.0 - (sum(similarities) / len(similarities))


def distinct_n(expanded_queries: list[str], n: int) -> float:
    """Compute distinct-n over all expanded queries as a simple diversity score."""
    all_ngrams: list[tuple[str, ...]] = []
    for query in expanded_queries:
        all_ngrams.extend(ngrams(tokenize(query, keep_stopwords=True), n))
    if not all_ngrams:
        return 0.0
    return len(set(all_ngrams)) / len(all_ngrams)


def englishish_score(expanded_queries: list[str]) -> float:
    """Approximate whether outputs are English search strings rather than Vietnamese text."""
    if not expanded_queries:
        return 0.0
    scores: list[float] = []
    for query in expanded_queries:
        chars = [c for c in query if not c.isspace()]
        if not chars:
            scores.append(0.0)
            continue
        ascii_ratio = sum(ord(c) < 128 for c in chars) / len(chars)
        vietnamese_ratio = sum(c in VIETNAMESE_DIACRITICS and ord(c) >= 128 for c in chars) / len(chars)
        scores.append(max(0.0, min(1.0, ascii_ratio - vietnamese_ratio)))
    return sum(scores) / len(scores)


def length_score(expanded_queries: list[str], min_words: int = 5, max_words: int = 18) -> float:
    """Reward query lengths that are useful for search without becoming verbose answers."""
    if not expanded_queries:
        return 0.0
    per_query: list[float] = []
    for query in expanded_queries:
        word_count = len(tokenize(query, keep_stopwords=True))
        if min_words <= word_count <= max_words:
            per_query.append(1.0)
        elif word_count < min_words:
            per_query.append(max(0.0, word_count / min_words))
        else:
            per_query.append(max(0.0, 1.0 - ((word_count - max_words) / max_words)))
    return sum(per_query) / len(per_query)


_EMBEDDER = None


def get_sentence_embedder(model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
    """Lazy-load the optional sentence-transformer used for semantic metrics."""
    global _EMBEDDER
    if _EMBEDDER is None:
        from sentence_transformers import SentenceTransformer

        _EMBEDDER = SentenceTransformer(model_name)
    return _EMBEDDER


def cosine(left: Iterable[float], right: Iterable[float]) -> float:
    """Compute cosine similarity without adding a heavy numerical dependency."""
    left_list = list(float(x) for x in left)
    right_list = list(float(x) for x in right)
    numerator = sum(a * b for a, b in zip(left_list, right_list))
    left_norm = math.sqrt(sum(a * a for a in left_list))
    right_norm = math.sqrt(sum(b * b for b in right_list))
    if left_norm == 0.0 or right_norm == 0.0:
        return 0.0
    return numerator / (left_norm * right_norm)


def optional_embedding_scores(expanded_queries: list[str], reference_expansions: list[str], enabled: bool = False) -> dict[str, float | None]:
    """Return optional semantic relevance/diversity scores when embeddings are enabled."""
    if not enabled:
        return {"embedding_relevance": None, "embedding_diversity": None}
    try:
        embedder = get_sentence_embedder()
        candidate_embeddings = embedder.encode(expanded_queries, normalize_embeddings=True)
        reference_embeddings = embedder.encode(reference_expansions, normalize_embeddings=True)

        relevance_scores: list[float] = []
        for candidate in candidate_embeddings:
            relevance_scores.append(max(cosine(candidate, ref) for ref in reference_embeddings))

        diversity_scores: list[float] = []
        for i, j in combinations(range(len(candidate_embeddings)), 2):
            diversity_scores.append(1.0 - cosine(candidate_embeddings[i], candidate_embeddings[j]))

        return {
            "embedding_relevance": float(sum(relevance_scores) / len(relevance_scores)) if relevance_scores else None,
            "embedding_diversity": float(sum(diversity_scores) / len(diversity_scores)) if diversity_scores else None,
        }
    except Exception as exc:
        print(f"Embedding metrics skipped: {exc}")
        return {"embedding_relevance": None, "embedding_diversity": None}


def compute_query_expansion_metrics(
    expanded_queries: list[str],
    case: dict[str, Any],
    k: int = DEFAULT_K,
    use_embeddings: bool = False,
) -> dict[str, Any]:
    """Compute all intrinsic H1 metrics for one generated query-expansion output."""
    expanded_queries = deduplicate_queries(expanded_queries)
    required_score, concept_hits = concept_coverage(expanded_queries, case["required_concepts"])
    forbidden_score, forbidden_hits = forbidden_avoidance(expanded_queries, case["forbidden_terms"])
    embedding = optional_embedding_scores(expanded_queries, case["reference_expansions"], enabled=use_embeddings)

    metrics: dict[str, Any] = {
        "exact_k": 1.0 if len(expanded_queries) == k else 0.0,
        "unique_ratio": len(set(q.casefold() for q in expanded_queries)) / max(1, len(expanded_queries)),
        "required_concept_coverage": required_score,
        "forbidden_avoidance": forbidden_score,
        "reference_token_overlap": reference_token_overlap(expanded_queries, case["reference_expansions"]),
        "pairwise_lexical_diversity": pairwise_jaccard_diversity(expanded_queries),
        "distinct_1": distinct_n(expanded_queries, 1),
        "distinct_2": distinct_n(expanded_queries, 2),
        "englishish_score": englishish_score(expanded_queries),
        "length_score": length_score(expanded_queries),
        "concept_hits": concept_hits,
        "forbidden_hits": forbidden_hits,
        **embedding,
    }

    # Weighted score is a ranking helper, not an absolute measure of retrieval quality.
    base_weights = {
        "exact_k": 0.12,
        "unique_ratio": 0.10,
        "required_concept_coverage": 0.24,
        "forbidden_avoidance": 0.16,
        "reference_token_overlap": 0.10,
        "pairwise_lexical_diversity": 0.10,
        "distinct_2": 0.08,
        "englishish_score": 0.05,
        "length_score": 0.05,
    }
    weighted_sum = sum(metrics[name] * weight for name, weight in base_weights.items())
    metrics["overall_score"] = weighted_sum / sum(base_weights.values())
    metrics["query_drift_risk"] = 1.0 - ((metrics["required_concept_coverage"] + metrics["forbidden_avoidance"]) / 2.0)
    return metrics


In [8]:
smoke_case = SYNTHETIC_CASES[0]
compute_query_expansion_metrics(smoke_case["reference_expansions"], smoke_case)

{'exact_k': 1.0,
 'unique_ratio': 1.0,
 'required_concept_coverage': 1.0,
 'forbidden_avoidance': 1.0,
 'reference_token_overlap': 1.0,
 'pairwise_lexical_diversity': 0.7890873015873016,
 'distinct_1': 0.6,
 'distinct_2': 0.9,
 'englishish_score': 1.0,
 'length_score': 1.0,
 'concept_hits': {'person': True,
  'clothing': True,
  'vehicle': True,
  'scene': True,
  'time': True},
 'forbidden_hits': [],
 'embedding_relevance': None,
 'embedding_diversity': None,
 'overall_score': 0.9709087301587301,
 'query_drift_risk': 0.0}

## 5. Models

This cell contains the core query-expansion runtime. It defines provider-agnostic model configs, generation hyperparameters, prompt variants, a Pydantic output schema, JSON parsing utilities, and a retry loop that tries to collect exactly `k` unique expanded queries.


In [9]:
load_dotenv(REPO_ROOT / ".env", override=True)
# print(os.getenv("GROQ_API_KEY"))
# print(os.getenv("LANGSMITH_API_KEY"))

True

Tạo sẵn class cho model config, parameters để thay đổi thông số model



In [10]:
@dataclass(frozen=True)
class ModelConfig:
    """Provider-agnostic model configuration used by the experiment grid."""

    alias: str
    provider: Literal["groq", "openai_compatible"]
    model_name: str
    api_key_env: str
    base_url_env: str | None = None
    supports_reasoning_effort: bool = False


@dataclass(frozen=True)
class GenerationParams:
    """Generation hyperparameters that can be swept in H1 experiments."""

    temperature: float = 0.4
    max_tokens: int = 2048
    timeout: int = 60
    max_retries: int = 2
    reasoning_effort: Literal["low", "medium", "high", "none", "default"] | None = "medium"
    output_mode: Literal["json_prompt", "structured"] = "json_prompt"


In [27]:
def load_model_configs(
    yaml_path: str | Path,
) -> dict[str, ModelConfig]:
    """
    Đọc cấu hình model từ file YAML.

    Args:
        yaml_path:
            Đường dẫn đến file YAML.

    Returns:
        Dictionary có dạng:
        {
            "model_alias": ModelConfig(...)
        }

    Raises:
        FileNotFoundError:
            Khi file YAML không tồn tại.
        ValueError:
            Khi cấu trúc YAML không hợp lệ.
    """
    yaml_path = Path(yaml_path)

    if not yaml_path.is_file():
        raise FileNotFoundError(
            f"Không tìm thấy file cấu hình: {yaml_path.resolve()}"
        )

    with yaml_path.open("r", encoding="utf-8") as file:
        raw_config: dict[str, Any] = yaml.safe_load(file) or {}

    raw_models = raw_config.get("models")

    if not isinstance(raw_models, dict):
        raise ValueError(
            "File YAML phải chứa một mapping có tên 'models'."
        )

    allowed_providers = {"groq", "openai_compatible"}
    model_configs: dict[str, ModelConfig] = {}

    for config_name, model_data in raw_models.items():
        if not isinstance(model_data, dict):
            raise ValueError(
                f"Cấu hình của model '{config_name}' phải là một mapping."
            )

        required_fields = {
            "alias",
            "provider",
            "model_name",
            "api_key_env",
        }

        missing_fields = required_fields - model_data.keys()

        if missing_fields:
            missing_text = ", ".join(sorted(missing_fields))
            raise ValueError(
                f"Model '{config_name}' thiếu các trường: {missing_text}"
            )

        provider = model_data["provider"]

        if provider not in allowed_providers:
            raise ValueError(
                f"Provider '{provider}' của model '{config_name}' "
                f"không hợp lệ. Các giá trị hỗ trợ: "
                f"{sorted(allowed_providers)}"
            )

        model_configs[config_name] = ModelConfig(
            alias=str(model_data["alias"]),
            provider=provider,
            model_name=str(model_data["model_name"]),
            api_key_env=str(model_data["api_key_env"]),
            base_url_env=model_data.get("base_url_env"),
            supports_reasoning_effort=bool(
                model_data.get("supports_reasoning_effort", False)
            ),
        )

    return model_configs

In [28]:
CONFIG_PATH =  os.path.join(EXPERIMENT_DIR, "config", "models.yaml")
print(CONFIG_PATH)
MODEL_CONFIGS: dict[str, ModelConfig] = load_model_configs(CONFIG_PATH)
MODEL_CONFIGS

D:\University\Projects\Individual projects\Multimodal-Retrieval\notebooks\agent\experiments\h1\config\models.yaml


{'gpt_oss_120b': ModelConfig(alias='gpt_oss_120b', provider='groq', model_name='openai/gpt-oss-120b', api_key_env='GROQ_API_KEY', base_url_env=None, supports_reasoning_effort=True),
 'gemma_4_31b': ModelConfig(alias='gemma_4_31b', provider='openai_compatible', model_name='google/gemma-4-31b-it', api_key_env='NVIDIA_API_KEY', base_url_env='NVIDIA_BASE_URL', supports_reasoning_effort=True),
 'nemotron_3_super_120b': ModelConfig(alias='nemotron_3_super_120b', provider='openai_compatible', model_name='nvidia/nemotron-3-super-120b-a12b', api_key_env='NVIDIA_API_KEY', base_url_env='NVIDIA_BASE_URL', supports_reasoning_effort=True),
 'nemotron_3_ultra_550b': ModelConfig(alias='nemotron_3_ultra_550b', provider='openai_compatible', model_name='nvidia/nemotron-3-ultra-550b-a55b', api_key_env='NVIDIA_API_KEY', base_url_env='NVIDIA_BASE_URL', supports_reasoning_effort=True),
 'qwen3.6-27b': ModelConfig(alias='qwen3.6-27b', provider='groq', model_name='qwen/qwen3.6-27b', api_key_env='GROQ_API_KEY',

In [29]:
pd.DataFrame([asdict(config) for config in MODEL_CONFIGS.values()])

,alias,provider,model_name,api_key_env,base_url_env,supports_reasoning_effort
0,gpt_oss_120b,groq,openai/gpt-oss-120b,GROQ_API_KEY,NaN,True
1,gemma_4_31b,openai_compatible,google/gemma-4-31b-it,NVIDIA_API_KEY,NVIDIA_BASE_URL,True
2,nemotron_3_super_120b,openai_compatible,nvidia/nemotron-3-super-120b-a12b,NVIDIA_API_KEY,NVIDIA_BASE_URL,True
3,nemotron_3_ultra_550b,openai_compatible,nvidia/nemotron-3-ultra-550b-a55b,NVIDIA_API_KEY,NVIDIA_BASE_URL,True
4,qwen3.6-27b,groq,qwen/qwen3.6-27b,GROQ_API_KEY,NaN,True
5,zai-glm-4.7,openai_compatible,zai-glm-4.7,CEREBRAS_API_KEY,CEREBRAS_BASE_URL,False


## 6. System prompt

In [30]:
SYSTEM_PROMPT_LIST: dict[str, str] = {
    "strict_multimedia_v1": '''
You are a query-expansion planner for a large-scale multimedia retrieval system.

The input query may be written in Vietnamese. Generate search-ready queries in English.

Requirements:
- Produce exactly {k} unique expanded queries.
- Preserve every reliable fact from the original query.
- Do not invent names, dates, organizations, people, locations, colors, scores, or events that are not explicitly present.
- Make every query self-contained so it can be searched independently.
- Diversify retrieval intent across visual evidence, OCR text, ASR/subtitles, event/action description, entities/relations, and concise keyword matching.
- Use natural English, not literal word-for-word translation.
- Do not answer the query.
- Return only valid JSON with this schema: {{"queries": ["query 1", "query 2"]}}.
'''.strip(),

    "keyword_control_v1": '''
You optimize Vietnamese-to-English query expansion for multimedia search.

Generate exactly {k} English queries. Each query must keep the original meaning but use a different search angle.

Good expansion types:
- visual objects, people, clothing, colors, scene, camera-visible actions
- OCR or text visible on screen when relevant
- ASR/subtitle/narration wording when relevant
- short keyword-style query for lexical search
- temporal sequence query for TRAKE-style prompts

Hard constraints:
- No new entities or unsupported details.
- No answers, explanations, bullet points, Markdown, or prose outside JSON.
- Every query must be independently understandable.
- Return JSON only: {{"queries": ["..."]}}.
'''.strip(),

    "minimal_translation_v1": '''
Translate and expand the user query into exactly {k} concise English search queries for multimedia retrieval.
Keep the facts unchanged, avoid hallucination, vary wording, and return only JSON: {{"queries": ["..."]}}.
'''.strip(),
}


## 7. Structured output

In [31]:
class QueryExpansionOutput(BaseModel):
    """Validated schema expected from every query-expansion model call."""

    queries: list[str] = Field(description="Unique, self-contained English search queries.")

    @field_validator("queries")
    @classmethod
    def clean_queries(cls, values: list[str]) -> list[str]:
        """Clean model output and fail fast if no usable query remains."""
        cleaned = deduplicate_queries(values)
        if not cleaned:
            raise ValueError("The model returned no usable queries.")
        return cleaned


def model_is_ready(config: ModelConfig) -> tuple[bool, str]:
    """Check whether required provider packages and environment variables are available."""
    if not os.getenv(config.api_key_env):
        return False, f"Missing {config.api_key_env}"
    if config.base_url_env and not os.getenv(config.base_url_env):
        return False, f"Missing {config.base_url_env}"
    if config.provider == "openai_compatible" and ChatOpenAI is None:
        return False, "Missing langchain-openai package"
    return True, "ready"


_LLM_CACHE: dict[tuple[Any, ...], Any] = {}


def make_llm(config: ModelConfig, params: GenerationParams):
    """Instantiate and cache a LangChain chat model for a model/hyperparameter pair."""
    ready, reason = model_is_ready(config)
    if not ready:
        raise RuntimeError(f"Model {config.alias} is not ready: {reason}")

    cache_key = (config.alias, config.provider, config.model_name, params.temperature, params.max_tokens, params.timeout, params.reasoning_effort)
    if cache_key in _LLM_CACHE:
        return _LLM_CACHE[cache_key]

    if config.provider == "groq":
        kwargs: dict[str, Any] = {
            "model": config.model_name,
            "temperature": params.temperature,
            "max_tokens": params.max_tokens,
            "timeout": params.timeout,
            "max_retries": params.max_retries,
        }
        if config.supports_reasoning_effort and params.reasoning_effort:
            kwargs["reasoning_effort"] = params.reasoning_effort
        llm = ChatGroq(**kwargs)
    elif config.provider == "openai_compatible":
        if ChatOpenAI is None:
            raise RuntimeError("Install langchain-openai to use OpenAI-compatible providers.")
        llm = ChatOpenAI(
            model=config.model_name,
            base_url=os.getenv(config.base_url_env or "OPENAI_BASE_URL"),
            api_key=os.getenv(config.api_key_env),
            temperature=params.temperature,
            max_tokens=params.max_tokens,
            timeout=params.timeout,
            max_retries=params.max_retries,
        )

    _LLM_CACHE[cache_key] = llm
    return llm


def extract_json_object(text: str) -> dict[str, Any]:
    """Extract the first valid JSON object from model text, tolerating fences and reasoning blocks."""
    text = text.strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE).strip()
        text = re.sub(r"```$", "", text.strip()).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError as original_error:
        decoder = json.JSONDecoder()
        for match in re.finditer(r"\{", text):
            try:
                value, _ = decoder.raw_decode(text[match.start():])
            except json.JSONDecodeError:
                continue
            if isinstance(value, dict):
                return value
        raise original_error


def parse_query_expansion_response(response: Any) -> QueryExpansionOutput:
    """Convert structured or raw LLM responses into QueryExpansionOutput."""
    if isinstance(response, QueryExpansionOutput):
        return response
    if isinstance(response, dict):
        return QueryExpansionOutput.model_validate(response)
    if isinstance(response, BaseMessage):
        content = response.content
    else:
        content = getattr(response, "content", response)
    if isinstance(content, list):
        content = "\n".join(str(item) for item in content)
    data = extract_json_object(str(content))
    return QueryExpansionOutput.model_validate(data)


def build_prompt(prompt_id: str) -> ChatPromptTemplate:
    """Build the system/human prompt pair for the selected prompt variant."""
    system_prompt = SYSTEM_PROMPT_LIST[prompt_id]
    return ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            (
                "human",
                '''Original query:\n{query}\n\nNumber of expanded queries required: {k}\n'''.strip(),
            ),
        ]
    )


def invoke_expander_once(
    query: str,
    k: int,
    model_config: ModelConfig,
    prompt_id: str,
    params: GenerationParams,
) -> tuple[QueryExpansionOutput, dict[str, Any]]:
    """Call one model once and return parsed output plus latency/usage metadata."""
    llm = make_llm(model_config, params)
    prompt = build_prompt(prompt_id)
    payload = {"query": query, "k": k}
    start = time.perf_counter()

    if params.output_mode == "structured":
        try:
            structured_llm = llm.with_structured_output(QueryExpansionOutput, method="json_schema", strict=True)
            response = (prompt | structured_llm).invoke(payload)
        except Exception:
            response = (prompt | llm).invoke(payload)
    else:
        response = (prompt | llm).invoke(payload)

    elapsed = time.perf_counter() - start
    usage = getattr(response, "usage_metadata", None)
    parsed = parse_query_expansion_response(response)
    return parsed, {"latency_sec": elapsed, "usage_metadata": usage}


def expand_query(
    query: str,
    k: int,
    model_config: ModelConfig,
    prompt_id: str,
    params: GenerationParams,
    max_attempts: int = 3,
) -> dict[str, Any]:
    """Generate exactly k unique expanded queries, retrying to fill missing items."""
    if not query.strip():
        raise ValueError("Query must not be empty.")
    if k < 1 or k > 20:
        raise ValueError("k must be between 1 and 20.")

    collected: list[str] = []
    usage_events: list[dict[str, Any]] = []
    total_latency = 0.0
    current_query = query.strip()

    for attempt in range(1, max_attempts + 1):
        result, run_info = invoke_expander_once(current_query, k, model_config, prompt_id, params)
        usage_events.append(run_info)
        total_latency += float(run_info["latency_sec"])
        collected = deduplicate_queries([*collected, *result.queries])
        if len(collected) >= k:
            return {
                "expanded_queries": collected[:k],
                "attempts": attempt,
                "latency_sec": total_latency,
                "usage_events": usage_events,
            }

        # Ask only for missing items on retry so duplicate-heavy models can recover.
        missing = k - len(collected)
        current_query = (
            f"{query.strip()}\n\n"
            "Already generated queries that must not be repeated:\n"
            + "\n".join(f"- {item}" for item in collected)
            + f"\nGenerate {missing} additional distinct English search queries."
        )

    return {
        "expanded_queries": collected[:k],
        "attempts": max_attempts,
        "latency_sec": total_latency,
        "usage_events": usage_events,
    }

## 8. Evaluation 

LangSmith is optional for H1. Use it when you want trace-level visibility into model calls and an experiment table that compares custom evaluators across model, prompt, and hyperparameter settings.



In [32]:
LANGSMITH_DATASET_NAME = os.getenv("LANGSMITH_H1_DATASET", "h1-query-expansion-synthetic-v5")

In [33]:

def require_langsmith_client() -> Any:
    """Create a LangSmith client after validating optional dependency and API key."""
    if Client is None:
        raise RuntimeError("Install langsmith to use LangSmith evaluation.")
    if not os.getenv("LANGSMITH_API_KEY"):
        raise RuntimeError("Set LANGSMITH_API_KEY before running LangSmith evaluation.")
    return Client()


def sync_langsmith_dataset(cases: list[dict[str, Any]] = SYNTHETIC_CASES, k: int = DEFAULT_K, dataset_name: str = LANGSMITH_DATASET_NAME):
    """Create or update the LangSmith dataset used for H1 query expansion eval."""
    client = require_langsmith_client()
    try:
        dataset = client.read_dataset(dataset_name=dataset_name)
    except Exception:
        dataset = client.create_dataset(
            dataset_name=dataset_name,
            description="Synthetic intrinsic evaluation set for H1 query expansion. Retrieval is intentionally excluded.",
        )

    existing_case_ids: set[str] = set()
    try:
        for example in client.list_examples(dataset_id=dataset.id):
            case_id = getattr(example, "inputs", {}).get("case_id")
            if case_id:
                existing_case_ids.add(case_id)
    except Exception:
        existing_case_ids = set()

    examples = []
    for case in cases:
        if case["case_id"] in existing_case_ids:
            continue
        examples.append(
            {
                "inputs": {
                    "case_id": case["case_id"],
                    "query": case["query"],
                    "query_type": case["query_type"],
                    "k": k,
                },
                "outputs": {
                    "required_concepts": case["required_concepts"],
                    "forbidden_terms": case["forbidden_terms"],
                    "reference_expansions": case["reference_expansions"],
                },
                "metadata": {"phase": "h1", "task": "query_expansion", "retrieval": False},
            }
        )

    if examples:
        client.create_examples(dataset_id=dataset.id, examples=examples)
        print(f"Added {len(examples)} new examples to {dataset_name}")
    else:
        print(f"Dataset {dataset_name} already has these case IDs.")
    return dataset


def case_from_langsmith(inputs: dict[str, Any], reference_outputs: dict[str, Any]) -> dict[str, Any]:
    """Reconstruct an H1 synthetic case from LangSmith example payloads."""
    return {
        "case_id": inputs["case_id"],
        "query_type": inputs.get("query_type", "unknown"),
        "query": inputs["query"],
        "required_concepts": reference_outputs["required_concepts"],
        "forbidden_terms": reference_outputs["forbidden_terms"],
        "reference_expansions": reference_outputs["reference_expansions"],
    }


def _langsmith_metrics(inputs: dict[str, Any], outputs: dict[str, Any], reference_outputs: dict[str, Any]) -> dict[str, Any]:
    """Shared metric adapter used by individual LangSmith evaluator functions."""
    case = case_from_langsmith(inputs, reference_outputs)
    expanded_queries = outputs.get("expanded_queries", [])
    return compute_query_expansion_metrics(expanded_queries, case=case, k=inputs.get("k", DEFAULT_K), use_embeddings=False)


def ls_overall_score(inputs: dict[str, Any], outputs: dict[str, Any], reference_outputs: dict[str, Any]) -> float:
    """LangSmith scalar evaluator for the weighted H1 score."""
    return float(_langsmith_metrics(inputs, outputs, reference_outputs)["overall_score"])


def ls_concept_coverage(inputs: dict[str, Any], outputs: dict[str, Any], reference_outputs: dict[str, Any]) -> float:
    """LangSmith scalar evaluator for required concept coverage."""
    return float(_langsmith_metrics(inputs, outputs, reference_outputs)["required_concept_coverage"])


def ls_forbidden_avoidance(inputs: dict[str, Any], outputs: dict[str, Any], reference_outputs: dict[str, Any]) -> float:
    """LangSmith scalar evaluator for hallucination/drift avoidance."""
    return float(_langsmith_metrics(inputs, outputs, reference_outputs)["forbidden_avoidance"])


def ls_diversity(inputs: dict[str, Any], outputs: dict[str, Any], reference_outputs: dict[str, Any]) -> float:
    """LangSmith scalar evaluator for lexical diversity."""
    return float(_langsmith_metrics(inputs, outputs, reference_outputs)["pairwise_lexical_diversity"])


def ls_exact_k(inputs: dict[str, Any], outputs: dict[str, Any], reference_outputs: dict[str, Any]) -> float:
    """LangSmith scalar evaluator for exact-k compliance."""
    return float(_langsmith_metrics(inputs, outputs, reference_outputs)["exact_k"])


LANGSMITH_EVALUATORS = [
    ls_overall_score,
    ls_concept_coverage,
    ls_forbidden_avoidance,
    ls_diversity,
    ls_exact_k,
]


def make_langsmith_target(model_alias: str, prompt_id: str, params: GenerationParams):
    """Wrap the local expander as a LangSmith target function."""
    model_config = MODEL_CONFIGS[model_alias]

    def _target(inputs: dict[str, Any]) -> dict[str, Any]:
        expansion = expand_query(
            query=inputs["query"],
            k=inputs.get("k", DEFAULT_K),
            model_config=model_config,
            prompt_id=prompt_id,
            params=params,
        )
        return {
            "expanded_queries": expansion["expanded_queries"],
            "attempts": expansion["attempts"],
            "latency_sec": expansion["latency_sec"],
            "model_alias": model_alias,
            "model_name": model_config.model_name,
            "prompt_id": prompt_id,
            "generation_params": asdict(params),
        }

    if traceable is None:
        return _target
    return traceable(name=f"h1_query_expansion/{model_alias}/{prompt_id}")(_target)


def run_langsmith_experiment(
    model_alias: str = "gpt_oss_120b",
    prompt_id: str = "strict_multimedia_v1",
    params: GenerationParams = GenerationParams(),
    dataset_name: str = LANGSMITH_DATASET_NAME,
    max_concurrency: int = 1,
):
    """Run one LangSmith experiment for a selected model/prompt configuration."""
    client = require_langsmith_client()
    dataset = sync_langsmith_dataset(k=DEFAULT_K, dataset_name=dataset_name)
    model_config = MODEL_CONFIGS[model_alias]
    ready, reason = model_is_ready(model_config)
    if not ready:
        raise RuntimeError(f"Model {model_alias} is not ready: {reason}")

    target = make_langsmith_target(model_alias, prompt_id, params)
    metadata = {
        "models": [model_config.model_name],
        "prompts": [prompt_id],
        "phase": "h1",
        "task": "query_expansion",
        "retrieval": False,
        "model_alias": model_alias,
        "provider": model_config.provider,
        "temperature": params.temperature,
        "max_tokens": params.max_tokens,
        "reasoning_effort": params.reasoning_effort,
        "output_mode": params.output_mode,
    }
    return client.evaluate(
        target,
        data=dataset.name,
        evaluators=LANGSMITH_EVALUATORS,
        experiment_prefix=f"h1-qe {model_alias} {prompt_id} t={params.temperature}",
        description="Intrinsic query expansion evaluation. No retrieval metrics are used.",
        max_concurrency=max_concurrency,
        metadata=metadata,
    )


## 9. Run LangSmith Experiment

This cell is also disabled by default. Set `RUN_LANGSMITH_EXPERIMENT = True` only after `LANGSMITH_API_KEY` and the required model provider keys are configured.


Ex1: gpt_oss_120b 
- model_alias="gpt_oss_120b",
- prompt_id="strict_multimedia_v1",
- params=GenerationParams(temperature=0.2, reasoning_effort="medium", output_mode="json_prompt"),
- max_concurrency=1,

Ex2:  nemotron_3_super_120b
- model_alias="nemotron_3_super_120b",
- prompt_id="strict_multimedia_v1",
- params=GenerationParams(temperature=0.2, reasoning_effort="medium", output_mode="json_prompt"),
- max_concurrency=1,


Ex3-4: gemma_4_31b
- model_alias="gemma_4_31b",
- prompt_id="strict_multimedia_v1",
- params=GenerationParams(temperature=0.2, reasoning_effort="medium", output_mode="json_prompt"),
- max_concurrency=1,

Ex5: qwen3.6-27b
- model_alias="qwen3.6-27b",
- prompt_id="strict_multimedia_v1",
- params=GenerationParams(temperature=0.2, reasoning_effort="none", output_mode="json_prompt"),
- max_concurrency=1,


In [48]:
RUN_LANGSMITH_EXPERIMENT = True

if RUN_LANGSMITH_EXPERIMENT:
    _LLM_CACHE.clear()
    ls_results = run_langsmith_experiment(
        model_alias="gpt_oss_120b",
        prompt_id="strict_multimedia_v1",
        params=GenerationParams(temperature=0.2, reasoning_effort="high", output_mode="json_prompt"),
        max_concurrency=1,
    )
    print(ls_results)
else:
    print("Set RUN_LANGSMITH_EXPERIMENT=True after configuring LANGSMITH_API_KEY and model keys.")


Dataset h1-query-expansion-synthetic-v5 already has these case IDs.
View the evaluation results for experiment: 'h1-qe gpt_oss_120b strict_multimedia_v1 t=0.2-cf6e1457' at:
https://smith.langchain.com/o/f70412f7-4c6b-4c3e-8ca7-23200eb8452d/datasets/b3df68f8-f92d-42d5-b05b-24883d079567/compare?selectedSessions=d2e4de56-b723-4087-95c0-715edd40e6ba




0it [00:00, ?it/s]

Error running target function: Expecting value: line 1 column 1 (char 0)
Traceback (most recent call last):
  File "C:\Users\Mario\AppData\Roaming\Python\Python314\site-packages\langsmith\evaluation\_runner.py", line 1990, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
    ~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Mario\AppData\Roaming\Python\Python314\site-packages\langsmith\run_helpers.py", line 777, in wrapper
    function_result = run_container["context"].run(
        func, *args, **kwargs
    )
  File "C:\Users\Mario\AppData\Local\Temp\ipykernel_39632\3270511974.py", line 117, in _target
    expansion = expand_query(
        query=inputs["query"],
    ...<3 lines>...
        params=params,
    )
  File "C:\Users\Mario\AppData\Local\Temp\ipykernel_39632\566020439.py", line 161, in expand_query
    result, run_info = invoke_expander_once(current_query, k, model_config, prompt_id, params)
                       ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^

<ExperimentResults h1-qe gpt_oss_120b strict_multimedia_v1 t=0.2-cf6e1457>


In [ ]:
llm2 = ChatGroq(
    model="qwen/qwen3.6-27b",
    api_key=os.environ["GROQ_API_KEY"],

    # Bật reasoning
    reasoning_effort="default", 

    # Tách reasoning khỏi final answer
    reasoning_format="parsed",

    temperature=0.2,
    max_tokens=2000,
    timeout=300,
    max_retries=3,
)

prompt = """
Original query:
Một người đàn ông đang chơi bóng rổ trong công viên.

Number of expanded queries required: 5
"""

llm_response = llm2.invoke(prompt)

reasoning = llm_response.additional_kwargs.get(
    "reasoning_content",
    ""
)

print("===== REASONING =====")
print(reasoning)

print("\n===== FINAL ANSWER =====")
print(llm_response.content)

===== REASONING =====
Here's a thinking process:

1.  **Analyze User Input:**
   - **Original Query:** "Một người đàn ông đang chơi bóng rổ trong công viên." (A man is playing basketball in the park.)
   - **Task:** Generate 5 expanded queries based on the original query.
   - **Language:** Vietnamese.

2.  **Understand "Expanded Queries":**
   - Expanded queries typically mean variations that add more detail, context, specificity, or different angles while preserving the core meaning.
   - They can include: different settings, times, weather, specific actions, demographics, equipment, atmosphere, or related activities.
   - Goal: Make them useful for search, image generation, or content creation, while staying relevant to the original.

3.  **Brainstorming Expansion Angles (Vietnamese):**
   - *Time/Weather:* buổi sáng sớm, chiều tà, nắng đẹp, mưa nhẹ, mùa hè
   - *Specific Action:* ném rổ, dribble, thi đấu 1vs1, tập luyện, ghi điểm
   - *Demographics/Details:* người đàn ông trung niê

In [42]:
llm_response

AIMessage(content='Dưới đây là 5 query mở rộng dựa trên câu gốc của bạn, được bổ sung thêm chi tiết về thời gian, bối cảnh, hành động và không khí để phù hợp với nhiều mục đích tìm kiếm hoặc tạo ảnh hơn:\n\n1. Một người đàn ông mặc áo thun thể thao đang dribble và ném bóng rổ trên sân ngoài trời trong công viên vào buổi sáng sớm.\n2. Hình ảnh một người đàn ông đang tập luyện bóng rổ một mình tại công viên trung tâm vào buổi chiều mùa hè.\n3. Một người đàn ông đang thi đấu bóng rổ với vài người bạn trên sân công viên, xung quanh có nhiều người đi dạo và ngồi xem.\n4. Cảnh một người đàn ông đang chơi bóng rổ giải trí trong công viên vào cuối tuần, trời nắng đẹp và có gió nhẹ.\n5. Một người đàn ông trung niên đang rèn luyện sức khỏe bằng cách chơi bóng rổ trên sân công viên gần hồ nước.\n\nNếu bạn cần các query theo hướng cụ thể hơn (ví dụ: tối ưu cho SEO, tạo ảnh AI, hay mô tả video), hãy cho mình biết để điều chỉnh phù hợp nhé!', additional_kwargs={'reasoning_content': 'Here\'s a thinki